In [71]:
import pandas as pd
from pathlib import Path
from PIL import Image
import shutil
import subprocess
import os
import re

PROJECT_ROOT = Path("/Users/bgifford/Documents/HOA/Ridge Park Reserve Study")
VARIANT_NAME = "2026_board_collab"
BASE_DIR = PROJECT_ROOT / VARIANT_NAME
ASSETS_DIR = PROJECT_ROOT / "assets"
SOURCE_DATA = BASE_DIR / "source_data"
REPORT_SOURCES_DIR = SOURCE_DATA / "report_sources"
WORKING_CSV = BASE_DIR / "working_csv"
REPORT_DIR = BASE_DIR / "report_latex"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COMPONENT_FILE = SOURCE_DATA / "component_list_v2.csv"
ASSUMPTIONS_FILE = SOURCE_DATA / "assumptions.csv"
ASSESSMENT_FILE = SOURCE_DATA / "assessment_contributions.csv"

ASSOCIATION_PROPERTIES_FILE = ASSETS_DIR / "association_properties.csv"
LATEX_TEMPLATE_FILE = ASSETS_DIR / "reserve_report_base.tex"
COVER_LETTER_FILE = REPORT_SOURCES_DIR / "cover_letter.txt"
PREPARER_REPORT_FILE = REPORT_SOURCES_DIR / "preparer_report.txt"
COVER_IMAGE_FILE = ASSETS_DIR / "cover_img.png"


In [72]:
REPORT_TITLE = "Reserve Management Plan"
REPORT_TYPE = "Type 1"
REPORT_SUBTITLE = "Reserve Study with Data-Driven Analysis"
SIGNER_NAME = "Brendan J. Gifford"
SIGNER_TITLE = "Ridge Park Treasurer"
REPORT_FILE_STEM = "ridge_park_reserve_report_latex"


In [73]:
def load_key_value_csv(path):
    df = pd.read_csv(path)
    key_col = df.columns[0]
    val_col = df.columns[1]
    return {str(k).strip(): str(v).strip() for k, v in zip(df[key_col], df[val_col])}


def latex_escape(value):
    s = '' if value is None else str(value)
    s = s.replace('\\', r'\textbackslash{}')
    for a, b in [('&', r'\&'), ('%', r'\%'), ('$', r'\$'), ('#', r'\#'), ('_', r'\_'), ('{', r'\{'), ('}', r'\}')]:
        s = s.replace(a, b)
    return s


def money(x, decimals=0):
    try:
        x = float(x)
    except Exception:
        return latex_escape(x)
    return f"\\${x:,.{decimals}f}"


def pct(x, decimals=2):
    return f"{float(x):.{decimals}f}\\%"


def ym_from_months(months):
    months = int(round(float(months)))
    y, m = divmod(months, 12)
    return f"{y}:{m:02d}"


def money_nodollar(x):
    return f"{float(x):,.0f}"


def format_short_date(dt):
    if pd.isna(dt):
        return ''
    return f"{dt.month}/{str(dt.year)[-2:]}"


def format_date_range(values):
    dates = pd.Series(values).dropna().sort_values()
    if len(dates) == 0:
        return ''
    if len(dates) == 1:
        return format_short_date(dates.iloc[0])
    return f"{format_short_date(dates.iloc[0])} - {format_short_date(dates.iloc[-1])}"


def text_to_latex_paragraphs(text):
    blocks = [b.strip() for b in re.split(r'\n\s*\n', str(text).strip()) if b.strip()]
    return '\n\n'.join(latex_escape(block) for block in blocks)


def parse_sectioned_text(text):
    text = str(text).strip().replace('\r\n', '\n')
    lines = text.split('\n')
    sections = []
    current_title = None
    current_body = []

    def flush():
        nonlocal current_title, current_body
        if current_title is not None:
            sections.append((current_title.strip(), '\n'.join(current_body).strip()))
        current_title = None
        current_body = []

    for line in lines:
        stripped = line.strip()

        if stripped.startswith('[') and stripped.endswith(']') and len(stripped) > 2:
            flush()
            current_title = stripped[1:-1].strip()
        elif stripped.startswith('## '):
            flush()
            current_title = stripped[3:].strip()
        elif stripped.startswith('# '):
            flush()
            current_title = stripped[2:].strip()
        elif stripped.endswith(':') and stripped[:-1].strip() and not current_body:
            flush()
            current_title = stripped[:-1].strip()
        else:
            if current_title is None and stripped:
                current_title = 'Section'
                current_body = [line]
            else:
                current_body.append(line)

    flush()
    return sections


def render_preparer_sections(text):
    parts = []
    for title, body in parse_sectioned_text(text):
        parts.append(rf"{{\bfseries {latex_escape(title)}}}\\")
        body_latex = text_to_latex_paragraphs(body)
        if body_latex:
            parts.append(body_latex)
        parts.append('')
    return '\n'.join(parts).strip()


def render_template(template_text, values):
    rendered = template_text
    for key, value in values.items():
        rendered = rendered.replace('{{' + key + '}}', str(value))
        rendered = rendered.replace('{' + key + '}', str(value))
    return rendered


def make_matrix_table_chunk(exp_matrix, start_col, end_col):
    years = list(exp_matrix.columns[start_col:end_col])
    first_col = exp_matrix.columns[0]

    header = "Category & " + " & ".join(str(y) for y in years) + r" \\"
    rows = [header, r"\midrule"]

    visible_idx = 0
    for _, row in exp_matrix.iterrows():
        vals = []
        for y in years:
            v = row[y]
            if pd.isna(v) or float(v) == 0:
                vals.append("")
            else:
                vals.append(money(v))

        prefix = r'\rowcolor{blue!12} ' if visible_idx % 2 == 0 else ''
        rows.append(prefix + latex_escape(row[first_col]) + " & " + " & ".join(vals) + r" \\")
        visible_idx += 1

    return "\n".join(rows)


In [74]:
assumptions = pd.read_csv(ASSUMPTIONS_FILE)
assumptions_map = dict(zip(assumptions['Parameter'], assumptions['Value']))
analysis_date = pd.to_datetime(assumptions_map['Analysis Date'])
analysis_date_str = analysis_date.strftime('%B %d, %Y').replace(' 0', ' ')
analysis_year = analysis_date.year

assoc = load_key_value_csv(ASSOCIATION_PROPERTIES_FILE)
ASSOC_NAME = assoc['ASSOC_NAME']
CITY_STATE = assoc['CITY_STATE']
PROJECT_TYPE = assoc['PROJECT_TYPE']
NUM_UNITS = assoc['NUM_UNITS']
CONSTRUCTION_DATE = assoc['CONSTRUCTION_DATE']
PREPARER = assoc['PREPARER']

statement = pd.read_csv(WORKING_CSV / 'statement_of_position_formatted.csv')
reserve = pd.read_csv(WORKING_CSV / 'reserve_projection.csv')
components = pd.read_csv(WORKING_CSV / 'component_list_detail.csv')
components['replacement_date'] = pd.to_datetime(components['replacement_date'], errors='coerce')
components['service_date'] = pd.to_datetime(components['service_date'], errors='coerce')
components['future_cost'] = pd.to_numeric(components['future_cost'], errors='coerce').fillna(0)
components['current_cost'] = pd.to_numeric(components['current_cost'], errors='coerce').fillna(0)
components['cost'] = pd.to_numeric(components['cost'], errors='coerce').fillna(0)
components['quantity'] = pd.to_numeric(components['quantity'], errors='coerce').fillna(0)
components['life_years'] = pd.to_numeric(components['life_years'], errors='coerce').fillna(0)
components['remaining_life_months'] = pd.to_numeric(components['remaining_life_months'], errors='coerce').fillna(0)

exp_matrix = pd.read_csv(WORKING_CSV / 'expenditures_matrix.csv')
exp_detail = pd.read_csv(WORKING_CSV / 'expenditures_by_year_detail.csv')
exp_detail['replacement_date'] = pd.to_datetime(exp_detail['replacement_date'], errors='coerce')
exp_detail['future_cost'] = pd.to_numeric(exp_detail['future_cost'], errors='coerce').fillna(0)

cover_img = REPORT_DIR / COVER_IMAGE_FILE.name
if COVER_IMAGE_FILE.exists():
    shutil.copy2(COVER_IMAGE_FILE, cover_img)

summary = (
    components.groupby('category', dropna=False)
    .agg(
        useful_min=('life_years', 'min'),
        useful_max=('life_years', 'max'),
        repl_min=('replacement_date', 'min'),
        repl_max=('replacement_date', 'max'),
        rem_min=('remaining_life_months', 'min'),
        rem_max=('remaining_life_months', 'max'),
        future_cost=('future_cost', 'sum'),
    )
    .reset_index()
    .sort_values('future_cost', ascending=False)
)
summary['useful_lives'] = summary.apply(
    lambda r: f"{int(r.useful_min)}" if int(r.useful_min) == int(r.useful_max) else f"{int(r.useful_min)}-{int(r.useful_max)}",
    axis=1
)
summary['replacement_years'] = summary.apply(
    lambda r: str(r.repl_min.year) if r.repl_min.year == r.repl_max.year else f"{r.repl_min.year}-{r.repl_max.year}",
    axis=1
)
summary['remaining_years'] = summary.apply(
    lambda r: ym_from_months(r.rem_min) if int(r.rem_min) == int(r.rem_max) else f"{ym_from_months(r.rem_min)}-{ym_from_months(r.rem_max)}",
    axis=1
)

upcoming = (
    exp_detail[exp_detail['replacement_date'].dt.year <= analysis_year + 1]
    .sort_values(['replacement_date', 'future_cost'], ascending=[True, False])
    .head(18)[['replacement_date', 'category', 'component', 'future_cost']]
)

component_summary = components.copy()
for col in ['category', 'component', 'quantity_units', 'cost_units']:
    if col in component_summary.columns:
        component_summary[col] = component_summary[col].fillna('').astype(str).str.strip()

component_summary['replace_date_display'] = (
    component_summary.groupby(['category', 'component'], dropna=False)['replacement_date']
    .transform(format_date_range)
)

component_summary_grouped = (
    component_summary.groupby(['category', 'component'], dropna=False)
    .agg(
        replace_date_display=('replace_date_display', 'first'),
        basis_cost=('cost', 'first'),
        quantity=('quantity', 'sum'),
        current_cost=('current_cost', 'sum'),
        life_years_min=('life_years', 'min'),
        life_years_max=('life_years', 'max'),
        rem_life_min=('remaining_life_months', 'min'),
        rem_life_max=('remaining_life_months', 'max'),
        future_cost=('future_cost', 'sum'),
        replacement_date_min=('replacement_date', 'min'),
    )
    .reset_index()
    .sort_values(['category', 'replacement_date_min', 'component'])
)

component_summary_grouped['quantity_display'] = component_summary_grouped['quantity'].apply(
    lambda x: f"{float(x):,.0f}" if float(x).is_integer() else f"{float(x):,.2f}"
)

component_summary_grouped['est_life_display'] = component_summary_grouped.apply(
    lambda r: (
        f"{int(r.life_years_min)}"
        if int(r.life_years_min) == int(r.life_years_max)
        else f"{int(r.life_years_min)}-{int(r.life_years_max)}"
    ),
    axis=1
)

component_summary_grouped['rem_life_display'] = component_summary_grouped.apply(
    lambda r: (
        ym_from_months(r.rem_life_min)
        if int(r.rem_life_min) == int(r.rem_life_max)
        else f"{ym_from_months(r.rem_life_min)}-{ym_from_months(r.rem_life_max)}"
    ),
    axis=1
)

ROW = r' \\'


def rows_join(rows):
    return '\n'.join(rows)


def make_statement_table(df):
    rows = []
    for i, r in enumerate(df.itertuples()):
        prefix = r'\rowcolor{blue!12} ' if i % 2 == 0 else ''
        rows.append(f"{prefix}{latex_escape(r.metric)} & {latex_escape(r.formatted)}{ROW}")
    return rows_join(rows)


def make_summary_table(df):
    rows = []
    for i, r in enumerate(df.itertuples()):
        prefix = r'\rowcolor{blue!12} ' if i % 2 == 0 else ''
        rows.append(
            f"{prefix}{latex_escape(r.category)} & {latex_escape(r.useful_lives)} & "
            f"{latex_escape(r.replacement_years)} & {latex_escape(r.remaining_years)} & {money(r.future_cost)}{ROW}"
        )
    return rows_join(rows)


def make_percent_funded_table(df):
    rows = []
    for i, r in enumerate(df.itertuples()):
        prefix = r'\rowcolor{blue!12} ' if i % 2 == 0 else ''
        rows.append(
            f"{prefix}{int(r.year)} & {money(r.end_balance)} & {money(r.funded_balance)} & {pct(r.percent_funded)}{ROW}"
        )
    return rows_join(rows)


def make_cashflow_table(df):
    rows = []
    for i, r in enumerate(df.itertuples()):
        prefix = r'\rowcolor{blue!12} ' if i % 2 == 0 else ''
        rows.append(
            f"{prefix}{int(r.year)} & {money(r.begin_balance)} & {money(r.contribution)} & "
            f"{money(r.special_assessment)} & {money(r.expenditures)} & {money(r.interest)} & {money(r.end_balance)}{ROW}"
        )
    return rows_join(rows)


def make_component_summary_longtable(df):
    rows = []
    shade_idx = 0

    for category, sub in df.groupby('category', sort=False, dropna=False):
        category_label = '' if pd.isna(category) else str(category)
        rows.append(rf"\multicolumn{{8}}{{l}}{{\textbf{{{latex_escape(category_label)}}}}}{ROW}")

        for _, r in sub.iterrows():
            prefix = r'\rowcolor{blue!12} ' if shade_idx % 2 == 0 else ''
            shade_idx += 1
            rows.append(
                f"{prefix}{latex_escape(r['component'])} & {latex_escape(r['replace_date_display'])} & "
                f"{money(r['basis_cost'])} & {latex_escape(r['quantity_display'])} & "
                f"{money(r['current_cost'])} & {latex_escape(r['est_life_display'])} & "
                f"{latex_escape(r['rem_life_display'])} & {money(r['future_cost'])}{ROW}"
            )

        rows.append(
            rf"\addlinespace[1pt] \multicolumn{{4}}{{r}}{{}} & \textbf{{{money(sub['current_cost'].sum())}}} & & & \textbf{{{money(sub['future_cost'].sum())}}}{ROW}"
        )
        rows.append(r"\addlinespace[6pt]")

    return rows_join(rows)


def make_upcoming_table(df):
    rows = []
    for i, r in enumerate(df.itertuples(index=False)):
        prefix = r'\rowcolor{blue!12} ' if i % 2 == 0 else ''
        rows.append(
            f"{prefix}{r.replacement_date.strftime('%Y-%m-%d')} & {latex_escape(r.category)} & "
            f"{latex_escape(r.component)} & {money(r.future_cost)}{ROW}"
        )
    return rows_join(rows)


cover_letter_body = text_to_latex_paragraphs(COVER_LETTER_FILE.read_text())
preparer_body = render_preparer_sections(PREPARER_REPORT_FILE.read_text())
template_text = LATEX_TEMPLATE_FILE.read_text()

template_values = {
    'ASSOC_NAME': latex_escape(ASSOC_NAME),
    'CITY_STATE': latex_escape(CITY_STATE),
    'PROJECT_TYPE': latex_escape(PROJECT_TYPE),
    'NUM_UNITS': latex_escape(NUM_UNITS),
    'CONSTRUCTION_DATE': latex_escape(CONSTRUCTION_DATE),
    'PREPARER': latex_escape(PREPARER),
    'ANALYSIS_DATE_STR': analysis_date_str,
    'ANALYSIS_YEAR': analysis_year,
    'PROJECTION_END_YEAR': int(reserve['year'].max()),
    'REPORT_TITLE': latex_escape(REPORT_TITLE),
    'REPORT_TYPE': latex_escape(REPORT_TYPE),
    'REPORT_SUBTITLE': latex_escape(REPORT_SUBTITLE),
    'SIGNER_NAME': latex_escape(SIGNER_NAME),
    'SIGNER_TITLE': latex_escape(SIGNER_TITLE),
    'COVER_IMAGE_FILENAME': cover_img.name,
    'COVER_LETTER_BODY': cover_letter_body,
    'PREPARER_BODY': preparer_body,
    'STATEMENT_TABLE': make_statement_table(statement),
    'SUMMARY_TABLE': make_summary_table(summary),
    'PERCENT_FUNDED_TABLE': make_percent_funded_table(reserve),
    'CASHFLOW_TABLE': make_cashflow_table(reserve),
    'MATRIX_1_10': make_matrix_table_chunk(exp_matrix, 1, 11),
    'MATRIX_11_20': make_matrix_table_chunk(exp_matrix, 11, 21),
    'MATRIX_21_30': make_matrix_table_chunk(exp_matrix, 21, 31),
    'COMPONENT_SUMMARY_LONGTABLE': make_component_summary_longtable(component_summary_grouped),
    'UPCOMING_TABLE': make_upcoming_table(upcoming),
    'FIRST_YEAR_CONTRIBUTION': money(reserve.iloc[0]['contribution']),
    'FIRST_YEAR_END_BALANCE': money(reserve.iloc[0]['end_balance']),
    'FINAL_YEAR': int(reserve.iloc[-1]['year']),
    'FINAL_YEAR_END_BALANCE': money(reserve.iloc[-1]['end_balance']),
    'INFLATION_PCT': f"{float(assumptions_map['Inflation']) * 100:.2f}\\%",
    'INVESTMENT_PCT': f"{float(assumptions_map['Investment']) * 100:.2f}\\%",
    'CONTRIBUTION_FACTOR': f"{float(assumptions_map['Contribution Factor']):.2f}",
    'BEGIN_BALANCE': money(float(assumptions_map['Begin Balance'])),
    'MAX_CATEGORY': latex_escape(summary.iloc[0]['category']),
    'MAX_CATEGORY_COST': money(summary.iloc[0]['future_cost']),
    'MIN_BALANCE_YEAR': int(reserve.loc[reserve['end_balance'].idxmin(), 'year']),
    'MIN_BALANCE_VALUE': money(reserve['end_balance'].min()),
    'MAX_FUNDED_YEAR': int(reserve.loc[reserve['percent_funded'].idxmax(), 'year']),
    'MAX_FUNDED_VALUE': pct(reserve['percent_funded'].max()),
}

tex = render_template(template_text, template_values)

tex_path = REPORT_DIR / f'{REPORT_FILE_STEM}.tex'
tex_path.write_text(tex)
print(tex_path)


/Users/bgifford/Documents/HOA/Ridge Park Reserve Study/2026_board_collab/report_latex/ridge_park_reserve_report_latex.tex


In [75]:
from pathlib import Path
import subprocess
import shutil
import os

# ------------------------------------------------------------
# COMPILE LATEX TO PDF (MacTeX path fix)
# ------------------------------------------------------------

TEX_FILE = REPORT_DIR / "ridge_park_reserve_report_latex.tex"
PDF_FILE = REPORT_DIR / "ridge_park_reserve_report_latex.pdf"
BASE_NAME = TEX_FILE.stem

if not TEX_FILE.exists():
    raise FileNotFoundError(f"TeX file not found: {TEX_FILE}")

# Force MacTeX bin onto PATH for this Python process
mactex_bin = "/Library/TeX/texbin"
env = os.environ.copy()
env["PATH"] = mactex_bin + os.pathsep + env.get("PATH", "")

def find_executable(name):
    path = shutil.which(name, path=env["PATH"])
    if path:
        return path
    candidate = Path(mactex_bin) / name
    if candidate.exists():
        return str(candidate)
    return None

latexmk = find_executable("latexmk")
pdflatex = find_executable("pdflatex")

print("PATH =", env["PATH"])
print("latexmk =", latexmk)
print("pdflatex =", pdflatex)

if not latexmk and not pdflatex:
    raise OSError(
        "Still cannot find latexmk or pdflatex. "
        "Restart Jupyter/VS Code completely after installing MacTeX."
    )

def run_cmd(cmd, cwd):
    print(f"\nRunning: {' '.join(cmd)}\n")
    result = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        log_file = cwd / f"{BASE_NAME}.log"
        if log_file.exists():
            print("\n" + "=" * 80)
            print("LAST 120 LINES OF LOG")
            print("=" * 80)
            log_lines = log_file.read_text(errors="ignore").splitlines()
            print("\n".join(log_lines[-120:]))
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {' '.join(cmd)}"
        )
    return result

# Clean stale files
if latexmk:
    run_cmd([latexmk, "-C", TEX_FILE.name], cwd=REPORT_DIR)

# Build
if latexmk:
    run_cmd(
        [latexmk, "-g", "-pdf", "-interaction=nonstopmode", "-halt-on-error", TEX_FILE.name],
        cwd=REPORT_DIR,
    )
else:
    run_cmd(
        [pdflatex, "-interaction=nonstopmode", "-halt-on-error", TEX_FILE.name],
        cwd=REPORT_DIR,
    )
    run_cmd(
        [pdflatex, "-interaction=nonstopmode", "-halt-on-error", TEX_FILE.name],
        cwd=REPORT_DIR,
    )

if not PDF_FILE.exists():
    raise FileNotFoundError(f"Compilation finished, but PDF was not found: {PDF_FILE}")

print(f"\nPDF created successfully:\n{PDF_FILE}")

PATH = /Library/TeX/texbin:/Users/bgifford/miniconda3/envs/ridgepark_reserve/bin:/Users/bgifford/miniconda3/condabin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/opt/pmk/env/global/bin:/Library/TeX/texbin
latexmk = /Library/TeX/texbin/latexmk
pdflatex = /Library/TeX/texbin/pdflatex

Running: /Library/TeX/texbin/latexmk -C ridge_park_reserve_report_latex.tex

Rc files read:
  NONE
Latexmk: This is Latexmk, John Collins, 15 June 2025. Version 4.87.
No specific requests made, so using default for latexmk.
Latexmk: Doing full clean up for 'ridge_park_reserve_report_latex.tex'



Running: /Library/TeX/texbin/latexmk -g -pdf -interaction=nonstopmode -halt-on-error ridge_park_reserve_report_latex.tex

Rc files read:
  NONE
Latexmk: This is La